# 05 · Full GUE Matrix Simulation: Zeta vs Poisson vs Random Hermitian Spectra

This notebook upgrades the GUE comparison by generating **actual random Hermitian matrices** and comparing their eigenvalue spacings with zeta-zero spacings.

Earlier notebooks used the Wigner surmise as a GUE-style proxy.  
Here we instead:

1. generate random complex Hermitian matrices
2. compute eigenvalues
3. extract bulk spacings
4. locally normalize the spacing sequences
5. compare zeta, Poisson, and matrix-derived GUE statistics
6. compare the unfolding-free ratio statistic

## Why this matters

This moves the repo from a GUE-style analytic proxy to a direct numerical random-matrix reference.

The notebook is still finite and exploratory:
- zeta zeros come from `mpmath`
- GUE is approximated numerically through finite random matrices
- spacings are taken from the bulk of the spectrum
- local normalization is used for spacing histograms, while the ratio statistic remains unfolding-free

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Zeta zeros and spacing sequence

In [ ]:
N = 500
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

len(t), len(zeta_gaps), zeta_gaps[:5]

## Local normalization helper

For spacing histograms we use a simple moving-average local normalization.
This is a lightweight unfolding step.

In [ ]:
def local_normalize_gaps(gaps, window=15):
    kernel = np.ones(window) / window
    local_means = np.convolve(gaps, kernel, mode="same")
    half = window // 2
    for i in range(half):
        local_means[i] = gaps[:i + half + 1].mean()
        local_means[-(i + 1)] = gaps[-(i + half + 1):].mean()
    return gaps / local_means

In [ ]:
zeta_local = local_normalize_gaps(zeta_gaps, window=15)
zeta_local.mean(), zeta_local.std()

## Poisson control

Independent exponential spacings with mean 1.

In [ ]:
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))
poisson_local = local_normalize_gaps(poisson_gaps, window=15)

poisson_local.mean(), poisson_local.std()

## Build a finite GUE reference from random Hermitian matrices

A standard finite GUE-style construction uses complex Hermitian matrices:

\[
H = \frac{1}{2}(A + A^\dagger),
\]

where \(A\) has independent complex Gaussian entries.

To reduce edge effects, we extract eigenvalue spacings from the **bulk** of each spectrum rather than from the full ordered list.

In [ ]:
def sample_gue_bulk_spacings(matrix_size=80, n_matrices=24, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []

    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0

        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=80, n_matrices=24, bulk_fraction=0.5, rng=rng)
len(gue_gaps), gue_gaps[:5]

In [ ]:
gue_local = local_normalize_gaps(gue_gaps, window=15)
gue_local.mean(), gue_local.std()

## Raw spacing snapshots

In [ ]:
plt.figure(figsize=(8.5, 4.8))
plt.plot(np.arange(1, 121), zeta_gaps[:120], label="zeta gaps")
plt.xlabel("index")
plt.ylabel("gap")
plt.title("Raw zeta gaps (first 120)")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.5, 4.8))
plt.plot(np.arange(1, 121), gue_gaps[:120], label="GUE bulk gaps")
plt.xlabel("index")
plt.ylabel("gap")
plt.title("Raw GUE bulk gaps (first 120)")
plt.legend()
plt.show()

## Locally normalized spacing comparison

In [ ]:
plt.figure(figsize=(8.8, 5.2))
bins = np.linspace(0, 3.5, 30)
plt.hist(zeta_local, bins=bins, density=True, alpha=0.55, label="zeta local norm")
plt.hist(poisson_local, bins=bins, density=True, alpha=0.45, label="Poisson local norm")
plt.hist(gue_local, bins=bins, density=True, alpha=0.45, label="GUE matrix local norm")
plt.xlabel("spacing")
plt.ylabel("density")
plt.title("Local-normalized spacing comparison")
plt.legend()
plt.show()

## Small-spacing comparison

This is where level repulsion should be most visible.

In [ ]:
small_bins = np.linspace(0, 1.5, 22)

plt.figure(figsize=(8.8, 5.0))
plt.hist(zeta_local, bins=small_bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_local, bins=small_bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_local, bins=small_bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel("spacing")
plt.ylabel("density")
plt.title("Small-spacing comparison")
plt.legend()
plt.show()

## Ratio statistic (unfolding-free)

We also compare the symmetrized ratio statistic

\[
\tilde r_n = \frac{\min(g_n,g_{n+1})}{\max(g_n,g_{n+1})}.
\]

This does not require local normalization.

In [ ]:
def ratio_stats(seq):
    g1 = seq[:-1]
    g2 = seq[1:]
    r = g2 / g1
    r_tilde = np.minimum(g1, g2) / np.maximum(g1, g2)
    return r, r_tilde

zeta_r, zeta_r_tilde = ratio_stats(zeta_gaps)
poisson_r, poisson_r_tilde = ratio_stats(poisson_gaps)
gue_r, gue_r_tilde = ratio_stats(gue_gaps)

zeta_r_tilde.mean(), poisson_r_tilde.mean(), gue_r_tilde.mean()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 1, 26)
plt.hist(zeta_r_tilde, bins=bins, density=True, alpha=0.55, label="zeta")
plt.hist(poisson_r_tilde, bins=bins, density=True, alpha=0.45, label="Poisson")
plt.hist(gue_r_tilde, bins=bins, density=True, alpha=0.45, label="GUE matrix")
plt.xlabel(r"$\tilde r_n = \min(g_n,g_{n+1})/\max(g_n,g_{n+1})$")
plt.ylabel("density")
plt.title("Symmetrized ratio comparison")
plt.legend()
plt.show()

## Mean symmetrized ratio by ensemble

In [ ]:
labels = ["zeta", "Poisson", "GUE matrix"]
means = [
    zeta_r_tilde.mean(),
    poisson_r_tilde.mean(),
    gue_r_tilde.mean(),
]

plt.figure(figsize=(7.4, 4.6))
plt.bar(labels, means)
plt.ylabel(r"mean symmetrized ratio $\langle \tilde r \rangle$")
plt.title("Mean unfolding-free ratio by ensemble")
plt.show()

means

## Local three-gap balance series for zeta

This is the same algebraic object as the symmetrized ratio, plotted as a running sequence.

In [ ]:
plt.figure(figsize=(8.4, 4.8))
plt.plot(np.arange(1, len(zeta_r_tilde) + 1), zeta_r_tilde, marker="o", linewidth=1)
plt.axhline(zeta_r_tilde.mean(), linestyle="--", linewidth=1, label=f"mean ≈ {zeta_r_tilde.mean():.3f}")
plt.xlabel("index")
plt.ylabel(r"$\tilde r_n$")
plt.title("Zeta unfolding-free local ratio series")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

## Summary statistics

In [ ]:
def summary_stats(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

summary = {
    "zeta_local_spacing": summary_stats(zeta_local),
    "poisson_local_spacing": summary_stats(poisson_local),
    "gue_local_spacing": summary_stats(gue_local),
    "zeta_r_tilde": summary_stats(zeta_r_tilde),
    "poisson_r_tilde": summary_stats(poisson_r_tilde),
    "gue_r_tilde": summary_stats(gue_r_tilde),
}
summary

## Notes

- This notebook uses **actual finite random Hermitian matrices** rather than the Wigner-surmise shortcut.
- The GUE reference is still numerical and finite-size; it is not an asymptotic closed-form result.
- Bulk eigenvalue extraction helps reduce edge effects from the semicircle-law tails.
- The ratio statistic remains useful because it avoids explicit unfolding.

## Next directions

A natural next notebook could do one of these:

1. vary matrix size and number of matrices to test stability  
2. compare bulk-only and full-spectrum spacing statistics  
3. estimate a pair-correlation-style quantity  
4. test longer local descriptors built from 3, 4, or 5 consecutive gaps